# 10 — LLM judge probe

Before using an LLM on the hardest pairs (v7): real throughput on Kaggle's 2x T4 and zero-shot separation on labelled
training pairs. Positives: a true (S1, record) pair with up to 3 of the entity's other true records as context.
Negatives: same S1 and context, but a record of a *different* entity in the same country whose name shares the first word
(the look-alikes that cause false merges). Compared: Phi-3.5-mini-instruct (MIT, 3.8B) and Qwen2.5-1.5B-Instruct
(Apache-2.0, 1.5B).

In [ ]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
import numpy as np, pandas as pd
from data_io import load_split, load_pairs
from llm_judge import Judge, JudgeConfig, prompt
rng = np.random.default_rng(0)
s1, r = load_split("train"); pairs = load_pairs()
txt = lambda df: pd.Series((df["name"].fillna("") + " | " + df["address"].fillna("")).to_numpy(), index=df["entity_id"].astype(str).to_numpy())
S1T, RT = txt(s1), txt(r)
country = pd.Series(s1["country"].astype(str).to_numpy(), index=s1["entity_id"].astype(str).to_numpy())
own = pairs.groupby("s1")["r"].agg(list)
ents = rng.choice(own.index.to_numpy(), 3000, replace=False)
r_first = pd.Series(r["name"].fillna("").str.lower().str.extract(r"^(?P<w>\w+)")["w"].to_numpy(), index=RT.index)
r_country = pd.Series(r["country"].astype(str).to_numpy(), index=RT.index)
owner = pairs.set_index("r")["s1"]
by_key = pd.DataFrame({"id": RT.index, "key": r_country.to_numpy() + "|" + r_first.fillna("").to_numpy()}).groupby("key")["id"].agg(list)
rows = []
for e in ents:
    recs = own[e]
    cand = recs[0]; sibs = recs[1:4]
    first = (s1.set_index(s1["entity_id"].astype(str)).at[e, "name"] or "").lower().split()
    key = country[e] + "|" + (first[0] if first else "")
    pool = [x for x in by_key.get(key, [])[:200] if owner.get(x) != e]
    if not pool: continue
    neg = pool[rng.integers(len(pool))]
    rows.append((e, cand, sibs, 1)); rows.append((e, neg, sibs, 0))
df = pd.DataFrame(rows, columns=["s1", "r", "sibs", "y"])
print(len(df), "labelled pairs", df.y.mean())

In [ ]:
from sklearn.metrics import roc_auc_score
res = {}
for name, batch in (("microsoft/Phi-3.5-mini-instruct", 32), ("Qwen/Qwen2.5-1.5B-Instruct", 64)):
    cfg = JudgeConfig(model=name, batch=batch)
    j = Judge(cfg)
    P = [prompt(S1T[a], [RT[x] for x in s], RT[b], cfg) for a, b, s in zip(df.s1, df.r, df.sibs)]
    t = time.time(); sc = j.score(P); dt = time.time() - t
    res[name] = {"pairs_per_s": len(P) / dt, "auc": roc_auc_score(df.y, sc),
                 "acc@0.5": float(((sc >= 0.5) == df.y).mean()),
                 "precision@0.9": float(df.y[sc >= 0.9].mean()) if (sc >= 0.9).any() else None,
                 "share>=0.9": float((sc >= 0.9).mean()), "npv@0.1": float(1 - df.y[sc <= 0.1].mean()) if (sc <= 0.1).any() else None,
                 "share<=0.1": float((sc <= 0.1).mean())}
    print(name, res[name])
    df[name] = sc
    del j
    import torch, gc; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(res).T

In [ ]:
# where each model is wrong (examples)
name = "microsoft/Phi-3.5-mini-instruct"
bad = df[((df[name] >= 0.5) != (df.y == 1))].head(12)
for a, b, s, y, sc in zip(bad.s1, bad.r, bad.sibs, bad.y, bad[name]):
    print(f"y={y} score={sc:.2f}\n  S1  {S1T[a]}\n  rec {RT[b]}\n  sib {[RT[x] for x in s][:2]}")